# A sales analytics pipeline with `%cash_on`

This notebook builds a small sales pipeline: generate data, join it, add
features, aggregate, plot. Each slow step sleeps briefly to stand in for real
work. It needs pandas, numpy and matplotlib.

Run it top to bottom once. Then run any cell again: its slow statements are
restored instead of recomputed. Restart the kernel and run everything again:
the slow results come back from disk.

**Read the badge, not the clock.** cash prints a badge under every cell it runs.
Open it to see one row per statement: **CACHED** (green) means cash restored the
value instead of running the code, **EXECUTED** (ochre) means the code ran.
Printed output is no evidence either way: a restored statement replays what it
printed.

In [ ]:
import cash
%cash_on

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Generate the data

Each table comes from its own seeded generator, so every run produces the same
data and the cached values match what a fresh run would give.

In [ ]:
def make_sales(n, seed):
    time.sleep(0.5)  # stands in for a slow load
    rng = np.random.default_rng(seed)
    sales = pd.DataFrame(
        {
            "transaction_id": range(1, n + 1),
            "date": pd.date_range("2024-01-01", "2024-12-31", periods=n),
            "customer_id": rng.integers(1000, 5000, n),
            "product": rng.choice(["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones", "Webcam", "Tablet", "Phone"], n),
            "quantity": rng.integers(1, 10, n),
            "unit_price": rng.uniform(10, 2000, n).round(2),
            "region": rng.choice(["North", "South", "East", "West"], n),
            "discount_percent": rng.choice([0, 5, 10, 15, 20], n),
        }
    )
    return sales.assign(total_amount=(sales.quantity * sales.unit_price * (1 - sales.discount_percent / 100)).round(2))


sales_df = make_sales(10_000, seed=42)
sales_df.head()

In [ ]:
def make_customers(n, seed):
    time.sleep(0.3)  # stands in for a slow load
    rng = np.random.default_rng(seed)
    return pd.DataFrame(
        {
            "customer_id": range(1000, 1000 + n),
            "join_date": pd.date_range("2020-01-01", periods=n),
            "customer_segment": rng.choice(["Premium", "Standard", "Basic"], n, p=[0.2, 0.5, 0.3]),
            "loyalty_points": rng.integers(0, 10_000, n),
        }
    )


customers_df = make_customers(4_000, seed=7)
customers_df.head()

In [ ]:
products_catalog = pd.DataFrame(
    [
        {"product": "Laptop", "category": "Electronics", "cost": 800},
        {"product": "Mouse", "category": "Accessories", "cost": 8},
        {"product": "Keyboard", "category": "Accessories", "cost": 30},
        {"product": "Monitor", "category": "Electronics", "cost": 300},
        {"product": "Headphones", "category": "Audio", "cost": 50},
        {"product": "Webcam", "category": "Electronics", "cost": 60},
        {"product": "Tablet", "category": "Electronics", "cost": 400},
        {"product": "Phone", "category": "Electronics", "cost": 600},
    ]
)
products_catalog

## 2. Join and add features

Each statement binds a new frame instead of changing one in place, so cash can
store every result.

In [ ]:
def enrich(sales, customers, catalog):
    time.sleep(0.8)  # stands in for an expensive join
    joined = sales.merge(customers, on="customer_id", how="left").merge(catalog, on="product", how="left")
    cost = joined["cost"] * joined["quantity"]
    return joined.assign(
        revenue=joined["total_amount"],
        profit=joined["total_amount"] - cost,
        profit_margin=((joined["total_amount"] - cost) / joined["total_amount"] * 100).round(2),
    )


enriched = enrich(sales_df, customers_df, products_catalog)
print(f"Total revenue: ${enriched['revenue'].sum():,.2f}")
enriched.head()

In [ ]:
enriched = enriched.assign(
    month=enriched["date"].dt.month,
    quarter=enriched["date"].dt.quarter,
    day_of_week=enriched["date"].dt.day_name(),
    is_weekend=enriched["date"].dt.dayofweek >= 5,
)
print(f"Weekend share: {enriched['is_weekend'].mean():.1%}")

## 3. Aggregate

In [ ]:
def customer_value(df):
    time.sleep(0.6)  # stands in for a heavy groupby
    clv = df.groupby("customer_id").agg(
        total_revenue=("revenue", "sum"),
        total_profit=("profit", "sum"),
        transactions=("transaction_id", "count"),
    )
    return clv.assign(avg_order_value=(clv.total_revenue / clv.transactions).round(2)).sort_values(
        "total_profit", ascending=False
    )


clv_metrics = customer_value(enriched)
clv_metrics.head(10)

In [ ]:
product_performance = (
    enriched.groupby(["product", "category"])
    .agg(total_revenue=("revenue", "sum"), total_profit=("profit", "sum"), units_sold=("quantity", "sum"))
    .reset_index()
    .sort_values("total_profit", ascending=False)
)
product_performance

## 4. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(product_performance["product"], product_performance["total_profit"])
ax.set_xlabel("Total profit ($)")
ax.set_title("Profit by product")
plt.show()

## 5. Change an input

**Try this:** set `selected_region` below to `"South"` and run the cell. The
badge says EXECUTED. Set it back to `"North"` and run it again: CACHED, because
cash still has the entry for `"North"`.

In [ ]:
def region_summary(df, region):
    time.sleep(0.5)  # stands in for a slow query
    rows = df[df["region"] == region]
    return {
        "revenue": round(rows["revenue"].sum(), 2),
        "profit": round(rows["profit"].sum(), 2),
        "transactions": len(rows),
        "customers": rows["customer_id"].nunique(),
    }


selected_region = "North"
region_stats = region_summary(enriched, selected_region)
region_stats

## 6. Turning caching off and on

`%cash_off` stops caching; cells run as plain Python until `%cash_on`.

In [ ]:
%cash_off

In [ ]:
# No badge: this cell runs as plain Python.
enriched.shape

In [ ]:
%cash_on

## 7. What cash did this session

To learn what cash caches and what it refuses, read
the [`%cash_on` guide](https://cash-lib.readthedocs.io/en/latest/notebook_caching_api/).

In [ ]:
%cash_stats